In [1]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

pd.set_option("display.max_rows", None) # Mostrar todas las filas
pd.set_option("display.max_columns", None) # Mostrar todas las columnas
# pd.set_option("display.max_colwidth", None) # Mostrar el contenido completo de cada celda (sin truncar)
# pd.set_option("display.width", 0) # Ajustar el ancho máximo de la salida al 100% (evita cortes con "…")
# pd.set_option("display.precision", 10) # Opcional: mayor precisión

In [ ]:
import os
import sys

project_path = os.path.abspath("Kronos")

sys.path.append(project_path)

print(project_path)
print(sys.path)

# # ============================================================
# # LOAD MODEL ONLY ONCE
# # ============================================================

# print("Loading Kronos model...")

# tokenizer = KronosTokenizer.from_pretrained(
#     "NeoQuasar/Kronos-tokenizer-base"
# )

# model = Kronos.from_pretrained(
#     "NeoQuasar/Kronos-base"
# )

# predictor = KronosPredictor(
#     model,
#     tokenizer,
#     max_context=512
# )

In [5]:
from pathlib import Path
project_dir = Path(__file__).parent.parent
project_dir
# output_file = project_dir / f"noticias_valora_{since_days}dias.json"

NameError: name '__file__' is not defined

In [ ]:
df_eval = pd.read_csv(r"..\\pfaval.csv")
df_eval.head()

,Price,Close,High,Low,Open,Volume
0,Ticker,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL
1,Date,NaN,NaN,NaN,NaN,NaN
2,2025-01-02,433.4523010253906,433.4523010253906,423.9882769855349,422.0954721775638,1242177
3,2025-01-03,438.184326171875,440.07713103654834,434.3987164425284,433.4523140101917,2385850
4,2025-01-07,440.0771179199219,440.0771179199219,430.6130938786332,438.18431311166415,1979583


# Modelo GRU (Gated Recurrent Unit)

## Parámetros del modelo

In [3]:
COLUMNA_OBJETIVO = "Precio cierre" # Variable a predecir
VENTANA = 30 # Días de historia que ve el modelo
HORIZONTE = 1 # Días hacia adelante a predecir
TEST_SIZE = 0.15 # 15 % de los datos para prueba
EPOCHS = 100
BATCH_SIZE = 16

## Preparar serie temporal

In [4]:
def preparar_datos(df, columna=COLUMNA_OBJETIVO, ventana=VENTANA):
    """Escala los precios y construye secuencias supervisadas (X, y)."""
    serie = df.sort_values("Fecha")[[columna]].dropna().values # (N, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    serie_scaled = scaler.fit_transform(serie)

    X, y = [], []
    for i in range(ventana, len(serie_scaled) - HORIZONTE + 1):
        X.append(serie_scaled[i - ventana : i])          # (ventana, 1)
        y.append(serie_scaled[i + HORIZONTE - 1])        # (1,)

    X = np.array(X)   # (muestras, ventana, 1)
    y = np.array(y)   # (muestras, 1)
    return X, y, scaler, serie

# Dividir los datos en entrenamiento y prueba

In [5]:
def dividir_datos(X, y, test_size=TEST_SIZE):
    corte = int(len(X) * (1 - test_size))
    return X[:corte], X[corte:], y[:corte], y[corte:]

## Construir modelo GRU

In [ ]:
def construir_modelo_gru(ventana=VENTANA):
    """Arquitectura GRU de dos capas (64 → 32 unidades) con Dropout + Dense para evitar sobreajuste."""
    modelo = Sequential([
    GRU(64, return_sequences=True, input_shape=(ventana, 1)),
    Dropout(0.2),
    GRU(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(HORIZONTE)
    ])
    modelo.compile(optimizer="adam", loss="mean_squared_error")
    modelo.summary()
    return modelo

## Entrenar

In [7]:
def entrenar_modelo(modelo, X_train, y_train):
    """entrenamiento con EarlyStopping para cortar automáticamente cuando deja de mejorar."""
    early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    historia = modelo.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
    )
    return historia

## Evaluar y graficar

In [8]:
def evaluar_y_graficar(modelo, X_test, y_test, scaler, df):
    """calcula MAE y RMSE, y grafica real vs predicción"""
    pred_scaled = modelo.predict(X_test)
    pred = scaler.inverse_transform(pred_scaled)
    real = scaler.inverse_transform(y_test)
    mae  = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    print(f"\n📊 MAE  : {mae:.2f} COP")
    print(f"📊 RMSE : {rmse:.2f} COP")

    fechas_test = df.sort_values("Fecha")["Fecha"].values[
        -len(real) - HORIZONTE + 1 : -HORIZONTE + 1 if HORIZONTE > 1 else None
    ]

    # Si vienen como (n, 1), los convertimos a 1D para graficar
    real_plot = real.ravel() if hasattr(real, "ravel") else real
    pred_plot = pred.ravel() if hasattr(pred, "ravel") else pred

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=fechas_test,
            y=real_plot,
            mode="lines",
            name="Precio real",
            line=dict(width=1.5)
        )
    )

    fig.add_trace(
        go.Scatter(
            x=fechas_test,
            y=pred_plot,
            mode="lines",
            name="Predicción GRU",
            line=dict(width=1.5, dash="dash")
        )
    )

    fig.update_layout(
        title="GRU - Predicción vs Precio real (PFAVAL)",
        xaxis_title="Fecha",
        yaxis_title="Precio cierre (COP)",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
    )

    fig.show()
    return pred, real

## Predecir próximo día

In [9]:
def predecir_proximo_dia(modelo, df, scaler, columna=COLUMNA_OBJETIVO, ventana=VENTANA):
    """Toma los últimos ventana cierres y predice el siguiente precio."""
    ultimos = df.sort_values("Fecha")[[columna]].dropna().values[-ventana:]
    ultimos_scaled = scaler.transform(ultimos).reshape(1, ventana, 1)
    pred_scaled = modelo.predict(ultimos_scaled, verbose=0)
    precio_futuro = scaler.inverse_transform(pred_scaled)[0, 0]
    ultima_fecha = df["Fecha"].max()
    print(f"\n🔮 Precio proyectado para el siguiente día hábil "
    f"(desde {ultima_fecha.date()}): {precio_futuro:.2f} COP")
    return precio_futuro

In [10]:
"""
Ejecuta el pipeline completo sobre df_eval y muestra la curva de aprendizaje, la gráfica de predicción vs real, 
y el precio proyectado para el siguiente día hábil.
"""

X, y, scaler, serie_original = preparar_datos(df_eval)
X_train, X_test, y_train, y_test = dividir_datos(X, y)

print(f"Muestras entrenamiento : {len(X_train)}")
print(f"Muestras prueba : {len(X_test)}")

modelo_gru = construir_modelo_gru()
historia = entrenar_modelo(modelo_gru, X_train, y_train)

# Curva de pérdida
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=list(range(1, len(historia.history["loss"]) + 1)),
        y=historia.history["loss"],
        mode="lines",
        name="Loss entrenamiento",
        line=dict(width=1.8)
    )
)

fig.add_trace(
    go.Scatter(
        x=list(range(1, len(historia.history["val_loss"]) + 1)),
        y=historia.history["val_loss"],
        mode="lines",
        name="Loss validación",
        line=dict(width=1.8, dash="dash")
    )
)

fig.update_layout(
    title="Curva de aprendizaje GRU",
    xaxis_title="Época",
    yaxis_title="MSE",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    height=350
)

fig.show()

predicciones, reales = evaluar_y_graficar(modelo_gru, X_test, y_test, scaler, df_eval)
predecir_proximo_dia(modelo_gru, df_eval, scaler)

Muestras entrenamiento : 182
Muestras prueba : 33


c:\Users\garriet\Documents\GitHub\Talleres_IA_2026\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 30, 64)         │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,817 (89.13 KB)

...

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 79ms/step - loss: 0.1285 - val_loss: 0.0104
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0282 - val_loss: 0.0048
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0177 - val_loss: 0.0129
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 0.0131 - val_loss: 0.0052
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 0.0088 - val_loss: 0.0053
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - loss: 0.0100 - val_loss: 0.0044
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - loss: 0.0093 - val_loss: 0.0055
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - loss: 0.0063 - val_loss: 0.0053
Epoch 9/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - loss: 0.0073 - val_loss: 0.0051
Epoch 10/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - loss: 0.0065 - val_loss: 0.0045
Epoch 11/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 0.0067 - val_loss: 0.0062
Epoch 12/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 664ms/step

📊 MAE  : 21.88 COP
📊 RMSE : 26.90 COP



🔮 Precio proyectado para el siguiente día hábil (desde 2026-05-26): 787.26 COP


np.float32(787.2557)

# Gráfica de velas

In [ ]:
# Soportes y resistencias + velas japonesas usando df_eval (Plotly)

def detectar_soportes_resistencias(
    df, ventana=3, tolerancia_pct=0.007, max_niveles=6,
    graficar=False, ultimas_barras=120
 ):
    """
    Detecta niveles por pivotes locales y agrupa precios cercanos.
    - ventana: barras a izquierda/derecha para definir pivote
    - tolerancia_pct: umbral de agrupacion (0.007 = 0.7%)
    - graficar: si True, muestra velas + niveles con Plotly
    """
    data = df.sort_values("Fecha").reset_index(drop=True).copy()

    lows = data["Precio mínimo"].values
    highs = data["Precio máximo"].values

    soportes_raw = []
    resistencias_raw = []

    for i in range(ventana, len(data) - ventana):
        low_win = lows[i - ventana : i + ventana + 1]
        high_win = highs[i - ventana : i + ventana + 1]

        if lows[i] == low_win.min():
            soportes_raw.append(float(lows[i]))
        if highs[i] == high_win.max():
            resistencias_raw.append(float(highs[i]))

    def agrupar_niveles(niveles):
        if not niveles:
            return []
        niveles = sorted(niveles)
        grupos = [[niveles[0]]]

        for nivel in niveles[1:]:
            media_grupo = sum(grupos[-1]) / len(grupos[-1])
            if abs(nivel - media_grupo) / media_grupo <= tolerancia_pct:
                grupos[-1].append(nivel)
            else:
                grupos.append([nivel])

        # Centro de cada grupo
        centros = [sum(g) / len(g) for g in grupos]
        return centros

    soportes = agrupar_niveles(soportes_raw)
    resistencias = agrupar_niveles(resistencias_raw)

    # Nos quedamos con niveles mas representativos en el rango de precios
    cierre_min = float(data["Precio cierre"].min())
    cierre_max = float(data["Precio cierre"].max())

    soportes = [s for s in soportes if s <= cierre_max]
    resistencias = [r for r in resistencias if r >= cierre_min]

    # Limitar cantidad para no saturar la grafica
    soportes = sorted(soportes)[-max_niveles:]
    resistencias = sorted(resistencias)[:max_niveles]

    if graficar:
        data_plot = data.tail(ultimas_barras).copy()
        data_plot["Open"] = data_plot["Precio cierre"].shift(1)
        data_plot.iloc[0, data_plot.columns.get_loc("Open")] = data_plot.iloc[0]["Precio cierre"]

        fig = go.Figure(
            data=[
                go.Candlestick(
                    x=data_plot["Fecha"],
                    open=data_plot["Open"],
                    high=data_plot["Precio máximo"],
                    low=data_plot["Precio mínimo"],
                    close=data_plot["Precio cierre"],
                    name="Velas"
                )
            ]
        )

        for s in soportes:
            fig.add_hline(y=s, line=dict(color="#1f77b4", width=1.1, dash="dash"), opacity=0.85)
        for r in resistencias:
            fig.add_hline(y=r, line=dict(color="#ff7f0e", width=1.1, dash="dash"), opacity=0.85)

        fig.update_layout(
            title="PFAVAL - Velas japonesas con soportes y resistencias",
            yaxis_title="Precio (COP)",
            xaxis_title="Fecha",
            xaxis_rangeslider_visible=False,
            template="plotly_white",
            height=650
        )
        fig.show()

    return soportes, resistencias


# Ejecutar sobre df_eval
soportes, resistencias = detectar_soportes_resistencias(
    df_eval,
    ventana=3,
    tolerancia_pct=0.007,
    max_niveles=6,
    graficar=True,
    ultimas_barras=120
)

print("Soportes detectados:", [round(s, 2) for s in soportes])
print("Resistencias detectadas:", [round(r, 2) for r in resistencias])

Soportes detectados: [742.0, 752.5, 759.5, 769.0, 778.0, 800.33]
Resistencias detectadas: [580.0, 591.6, 597.5, 660.0, 667.0, 680.0]


In [3]:
df_eval.tail()

,Fecha,Nemotécnico,Precio cierre,Precio máximo,Precio promedio ponderado,Precio mínimo,Variación absoluta,Variación porcentual,Cantidad,Volumen
249,2026-05-20,PFAVAL,779.0,780.0,NaN,759.0,18.0,2.37,1453018.0,1.118496e+09
250,2026-05-21,PFAVAL,780.0,785.0,NaN,767.0,1.0,0.13,610851.0,4.752776e+08
251,2026-05-22,PFAVAL,779.0,788.0,NaN,767.0,-1.0,-0.13,656376.0,5.114468e+08
252,2026-05-25,PFAVAL,792.0,794.0,NaN,774.0,13.0,1.67,722432.0,5.636740e+08
253,2026-05-26,PFAVAL,815.0,815.0,NaN,790.0,23.0,2.90,709081.0,5.664853e+08
